# Práctica de Laboratorio: Preprocesamiento y Limpieza de Datos para Machine Learning
**Asignatura:** Minería de Datos / Machine Learning  
**Modalidad:** Aprendizaje en contacto con el Docente  
**Herramienta:** Jupyter Notebook / JupyterLab  
**Estudiante:** Gino Maximiliano Bermúdez Santos  
**Dataset:** `dataset_limpieza_datos.csv` (10,000 registros, 8 columnas)  

---

## 1. Objetivo de la Práctica
Desarrollar habilidades prácticas en la preparación de datos crudos (*raw data*) para su posterior uso en modelos de entrenamiento de Machine Learning. Al finalizar, el estudiante será capaz de identificar anomalías y tratar el mismo conjunto de datos utilizando dos metodologías distintas de la industria:
1. **Enfoque 1 (Pandas):** Tratamiento manual, iterativo y exploratorio paso a paso.
2. **Enfoque 2 (Scikit-Learn Pipelines):** Automatización modular orientada a producción e inferencia en tiempo real mediante `Pipeline` y `ColumnTransformer`.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Configuración de visualización en Pandas
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


---
# Fase 1: Exploración Inicial (EDA Básico)

En esta fase se realiza una primera inspección del conjunto de datos crudo para evaluar dimensiones, tipos de datos, presencia de valores faltantes y anomalías numéricas o tipográficas.

In [2]:
# Carga del dataset desde la carpeta de datos
possible_paths = [
    "data/dataset_limpieza_datos.csv",
    "../data/dataset_limpieza_datos.csv",
    "apps/analytics/data/dataset_limpieza_datos.csv",
    "dataset_limpieza_datos.csv"
]

dataset_path = None
for p in possible_paths:
    if os.path.exists(p):
        dataset_path = p
        break

if not dataset_path:
    raise FileNotFoundError("No se encontró dataset_limpieza_datos.csv en las rutas configuradas.")

df_raw = pd.read_csv(dataset_path)
print(f"Dataset cargado exitosamente desde: {dataset_path}")
print(f"Dimensiones iniciales: {df_raw.shape[0]} registros x {df_raw.shape[1]} columnas.\n")
print("Primeros 5 registros:")
print(df_raw.head())

Dataset cargado exitosamente desde: data/dataset_limpieza_datos.csv
Dimensiones iniciales: 10000 registros x 8 columnas.

Primeros 5 registros:
   n  edad     genero     ingresos    altura     ciudad nivel_educacion  hijos
0  1  56.0   Femenino  80277.19986  1.813846      Quito             phd    3.0
1  2  69.0   Femenino  57084.76993  1.594719  Guayaquil      bachillers    4.0
2  3  46.0  Masculino  55146.56536  1.692878  Guayaquil       bachiller    0.0
3  4  32.0  Masculino -39848.81379  1.774996      Manta             NaN    2.0
4  5  60.0  Masculino  52134.05204  1.724397      Manta      bachiller     1.0


In [3]:
print("=== ESTRUCTURA DEL DATAFRAME (.info()) ===")
df_raw.info()

=== ESTRUCTURA DEL DATAFRAME (.info()) ===
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   n                10000 non-null  int64  
 1   edad             9533 non-null   float64
 2   genero           9592 non-null   str    
 3   ingresos         9527 non-null   float64
 4   altura           9731 non-null   float64
 5   ciudad           9158 non-null   str    
 6   nivel_educacion  9016 non-null   str    
 7   hijos            9587 non-null   float64
dtypes: float64(4), int64(1), str(3)
memory usage: 844.6 KB


In [4]:
print("=== CONTEO Y PORCENTAJE DE VALORES NULOS POR COLUMNA ===")
null_counts = df_raw.isnull().sum()
null_pct = (df_raw.isnull().mean() * 100).round(2)
null_summary = pd.DataFrame({"Nulos": null_counts, "Porcentaje (%)": null_pct})
print(null_summary)

=== CONTEO Y PORCENTAJE DE VALORES NULOS POR COLUMNA ===
                 Nulos  Porcentaje (%)
n                    0            0.00
edad               467            4.67
genero             408            4.08
ingresos           473            4.73
altura             269            2.69
ciudad             842            8.42
nivel_educacion    984            9.84
hijos              413            4.13


In [5]:
print("=== RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS (.describe()) ===")
print(df_raw.describe().round(2))

=== RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS (.describe()) ===
              n     edad    ingresos   altura    hijos
count  10000.00  9533.00     9527.00  9731.00  9587.00
mean    5000.50    49.96    70804.32     1.71     2.15
std     2886.90    25.35   137583.16     0.18     3.12
min        1.00   -20.00   -49555.25     1.34    -5.00
25%     2500.75    33.00    48790.66     1.63     1.00
50%     5000.50    49.00    59522.41     1.70     2.00
75%     7500.25    65.00    69945.33     1.77     3.00
max    10000.00   199.00  1991668.38     3.97    49.00


In [6]:
print("=== VALORES ÚNICOS EN VARIABLES CATEGÓRICAS ===")
for col in ["genero", "ciudad", "nivel_educacion"]:
    print(f"\n--- Conteo de categorías en '{col}': ---")
    print(df_raw[col].value_counts(dropna=False))

=== VALORES ÚNICOS EN VARIABLES CATEGÓRICAS ===

--- Conteo de categorías en 'genero': ---
genero
Masculino     4366
Femenino      4301
NaN            408
F              273
M              248
 femenino      205
masculino      199
Name: count, dtype: int64

--- Conteo de categorías en 'ciudad': ---
ciudad
Guayaquil        1926
Quito            1771
Esmeraldas       1739
Manta            1714
Santo Domingo    1657
NaN               842
Sto. Domingo      192
Esmeraldas        159
Name: count, dtype: int64

--- Conteo de categorías en 'nivel_educacion': ---
nivel_educacion
Secundaria        2144
NaN                984
bachillers         774
bachiller          753
Master             748
master             741
bachiller          732
phd                726
PhD                719
Ph.D               718
mre                700
sin  educacion     261
Name: count, dtype: int64


In [7]:
print("=== DIAGNÓSTICO DE VALORES NEGATIVOS E ILÓGICOS ===")
print(f"Edades negativas (< 0): {(df_raw['edad'] < 0).sum()} registros (Mínimo: {df_raw['edad'].min()})")
print(f"Edades irreales (> 100): {(df_raw['edad'] > 100).sum()} registros (Máximo: {df_raw['edad'].max()})")
print(f"Ingresos negativos (< 0): {(df_raw['ingresos'] < 0).sum()} registros (Mínimo: {df_raw['ingresos'].min()})")
print(f"Hijos negativos (< 0): {(df_raw['hijos'] < 0).sum()} registros (Mínimo: {df_raw['hijos'].min()})")
print(f"Hijos con valores decimales: {((df_raw['hijos'] % 1 != 0) & df_raw['hijos'].notna()).sum()} registros")

=== DIAGNÓSTICO DE VALORES NEGATIVOS E ILÓGICOS ===
Edades negativas (< 0): 176 registros (Mínimo: -20.0)
Edades irreales (> 100): 210 registros (Máximo: 199.0)
Ingresos negativos (< 0): 266 registros (Mínimo: -49555.24935)
Hijos negativos (< 0): 177 registros (Mínimo: -5.0)
Hijos con valores decimales: 288 registros


### Reporte de Diagnóstico Inicial sobre la Calidad del Dataset (Fase 1)

Tras la exploración inicial descriptiva, se identificaron las siguientes **5 anomalías principales**:

1. **Presencia Significativa de Valores Faltantes (NaN):**
   - 7 de las 8 columnas del dataset contienen valores nulos.
   - Las variables con mayor afectación son categóricas: `nivel_educacion` (984 nulos, ~9.84%) y `ciudad` (842 nulos, ~8.42%).
   - Las variables numéricas presentan aproximadamente entre un 2.7% y 4.7% de faltantes (`ingresos`: 473, `edad`: 467, `hijos`: 413, `genero`: 408, `altura`: 269).
2. **Valores Ilógicos y Negativos en Variables Cuantitativas:**
   - `edad`: Se detectaron **176 registros negativos** (mínimo: -20 años) y valores biológicamente imposibles superiores a 100 años (máximo: 199 años).
   - `ingresos`: Existen **266 registros con ingresos negativos** (mínimo: -\$49,555.25), lo cual viola la regla de dominio económico.
   - `hijos`: Se encontraron **177 registros con valores negativos** (mínimo: -5 hijos).
3. **Inconsistencia de Tipos y Naturaleza de Datos:**
   - La columna `hijos` tiene **288 registros con valores decimales** (por ejemplo, `1.7979` y `1.8558`), violando la naturaleza discreta del número de descendientes.
   - Al contener valores `NaN`, Pandas castea automáticamente las columnas enteras (`edad`, `hijos`) a `float64`.
4. **Inconsistencias Semánticas, Espacios Extra y Errores Tipográficos:**
   - `genero`: Espacios en blanco accidentales (`'masculino '`, `' femenino'`) y abreviaturas heterogéneas (`'M'`, `'F'`).
   - `ciudad`: Errores de normalización toponímica (`'Sto. Domingo'` vs `'Santo Domingo'`) y espacios residuales (`'Esmeraldas '`).
   - `nivel_educacion`: Múltiples variantes ortográficas para el mismo nivel educativo (`'phd'`, `'Ph.D'`, `'PhD'`, `'master'`, `'Master'`, `'mre'`, `'bachiller'`, `'bachillers'`, `'sin  educacion'`).
5. **Valores Atípicos Extremos (Outliers):**
   - En `ingresos`, el valor máximo alcanza **\$1,991,668.00**, alejándose drásticamente de la mediana (\$59,522.41) y la media (\$70,804.32), lo cual evidencia una fuerte asimetría positiva (*right-skewed*).
   - En `altura`, se observan alturas irreales de hasta **3.97 metros**.

---
# Fase 2: Enfoque 1 - Tratamiento Manual e Iterativo (Pandas)

En esta sección, se aplican las transformaciones paso a paso modificando directamente una copia del DataFrame (`df_manual`). Este enfoque es ideal para el análisis ad-hoc y la comprensión profunda de cada regla de negocio.

In [8]:
# 1. Copia independiente para el Enfoque 1
df_manual = df_raw.copy()

print("Conteo de anomalías antes de la corrección:")
print(f" - Edades < 0: {(df_manual['edad'] < 0).sum()}")
print(f" - Edades > 100: {(df_manual['edad'] > 100).sum()}")
print(f" - Ingresos < 0: {(df_manual['ingresos'] < 0).sum()}")
print(f" - Hijos < 0: {(df_manual['hijos'] < 0).sum()}")

# Reemplazo de valores negativos e ilógicos por NaN para forzar su imputación
df_manual.loc[df_manual['edad'] < 0, 'edad'] = np.nan
df_manual.loc[df_manual['edad'] > 100, 'edad'] = np.nan
df_manual.loc[df_manual['ingresos'] < 0, 'ingresos'] = np.nan
df_manual.loc[df_manual['hijos'] < 0, 'hijos'] = np.nan

# Corrección de hijos decimales: redondear al entero más cercano
df_manual.loc[df_manual['hijos'].notna(), 'hijos'] = df_manual.loc[df_manual['hijos'].notna(), 'hijos'].round()

print("\nVerificación tras reemplazo por NaN:")
print(f" - Edades < 0: {(df_manual['edad'] < 0).sum()}")
print(f" - Edades > 100: {(df_manual['edad'] > 100).sum()}")
print(f" - Ingresos < 0: {(df_manual['ingresos'] < 0).sum()}")
print(f" - Hijos < 0: {(df_manual['hijos'] < 0).sum()}")

Conteo de anomalías antes de la corrección:
 - Edades < 0: 176
 - Edades > 100: 210
 - Ingresos < 0: 266
 - Hijos < 0: 177

Verificación tras reemplazo por NaN:
 - Edades < 0: 0
 - Edades > 100: 0
 - Ingresos < 0: 0
 - Hijos < 0: 0


In [9]:
# 2. Mapeo y Normalización de Strings en variables categóricas

# A. Normalización de 'genero'
df_manual['genero'] = df_manual['genero'].astype(str).str.strip().str.lower()
mapa_genero = {
    'masculino': 'Masculino',
    'femenino': 'Femenino',
    'm': 'Masculino',
    'f': 'Femenino',
    'nan': np.nan,
    'none': np.nan,
    '': np.nan
}
df_manual['genero'] = df_manual['genero'].map(mapa_genero)

# B. Normalización de 'ciudad'
df_manual['ciudad'] = df_manual['ciudad'].astype(str).str.strip().str.title()
mapa_ciudad = {
    'Sto. Domingo': 'Santo Domingo',
    'Nan': np.nan,
    'None': np.nan,
    '': np.nan
}
df_manual['ciudad'] = df_manual['ciudad'].replace(mapa_ciudad)

# C. Normalización de 'nivel_educacion'
# Limpieza de espacios múltiples internos
df_manual['nivel_educacion'] = (
    df_manual['nivel_educacion']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(r'\s+', ' ', regex=True)
)
mapa_educacion = {
    'phd': 'Doctorado',
    'ph.d': 'Doctorado',
    'master': 'Maestría',
    'mre': 'Maestría',
    'bachiller': 'Bachillerato',
    'bachillers': 'Bachillerato',
    'secundaria': 'Secundaria',
    'sin educacion': 'Sin Educación',
    'nan': np.nan,
    'none': np.nan,
    '': np.nan
}
df_manual['nivel_educacion'] = df_manual['nivel_educacion'].map(mapa_educacion)

print("Categorías homologadas y unificadas:")
print("\nGénero:", df_manual['genero'].value_counts(dropna=False).to_dict())
print("\nCiudad:", df_manual['ciudad'].value_counts(dropna=False).to_dict())
print("\nNivel Educación:", df_manual['nivel_educacion'].value_counts(dropna=False).to_dict())

Categorías homologadas y unificadas:

Género: {'Masculino': 4813, 'Femenino': 4779, nan: 408}

Ciudad: {'Guayaquil': 1926, 'Esmeraldas': 1898, 'Santo Domingo': 1849, 'Quito': 1771, 'Manta': 1714, nan: 842}

Nivel Educación: {'Bachillerato': 2259, 'Maestría': 2189, 'Doctorado': 2163, 'Secundaria': 2144, nan: 984, 'Sin Educación': 261}


In [10]:
# 3. Imputación Manual de Valores Nulos

# Variables numéricas: Imputación con la MEDIANA
columnas_numericas = ['edad', 'ingresos', 'altura', 'hijos']
for col in columnas_numericas:
    mediana_val = df_manual[col].median()
    df_manual[col] = df_manual[col].fillna(mediana_val)
    print(f"Imputación numérica -> '{col}': mediana = {round(mediana_val, 2)}")

# Variables categóricas: Imputación con la MODA
columnas_categoricas = ['genero', 'ciudad', 'nivel_educacion']
for col in columnas_categoricas:
    moda_val = df_manual[col].mode().iloc[0]
    df_manual[col] = df_manual[col].fillna(moda_val)
    print(f"Imputación categórica -> '{col}': moda = '{moda_val}'")

# Casteo estricto a números enteros para 'edad' e 'hijos'
df_manual['edad'] = df_manual['edad'].astype(int)
df_manual['hijos'] = df_manual['hijos'].astype(int)

print("\nVerificación final de nulos en df_manual:", df_manual.isnull().sum().sum())
print("Tipos de datos resultantes:")
print(df_manual[['edad', 'hijos', 'ingresos', 'altura']].dtypes)

Imputación numérica -> 'edad': mediana = 49.0
Imputación numérica -> 'ingresos': mediana = 60065.87
Imputación numérica -> 'altura': mediana = 1.7
Imputación numérica -> 'hijos': mediana = 2.0
Imputación categórica -> 'genero': moda = 'Masculino'
Imputación categórica -> 'ciudad': moda = 'Guayaquil'
Imputación categórica -> 'nivel_educacion': moda = 'Bachillerato'

Verificación final de nulos en df_manual: 0
Tipos de datos resultantes:
edad          int64
hijos         int64
ingresos    float64
altura      float64
dtype: object


### Justificación Teórica: ¿Por qué usar la Mediana en lugar de la Media?

En presencia de distribuciones asimétricas o con **valores atípicos extremos (outliers)** (como los ingresos de hasta \$1,991,668.00 detectados en el dataset), la media aritmética se desplaza sensiblemente hacia la cola superior, sobrestimando el valor central típico. La **mediana** es una medida de tendencia central robusta (*resistente*), ya que representa el percentil 50 y no se ve afectada por la magnitud de observaciones extremas.

In [11]:
# 4. Estandarización Manual: Fórmula Z = (x - μ) / σ

def estandarizar_serie(serie: pd.Series) -> pd.Series:
    """Calcula la estandarización Z-score de forma manual y vectorizada."""
    mu = serie.mean()
    sigma = serie.std()
    return (serie - mu) / sigma

# Aplicar a la columna de ingresos
df_manual['ingresos_estandarizados'] = estandarizar_serie(df_manual['ingresos'])

# Verificación de propiedades estadísticas de Z (media ≈ 0, desviación estándar ≈ 1)
z_mean = df_manual['ingresos_estandarizados'].mean()
z_std = df_manual['ingresos_estandarizados'].std()

print("=== VERIFICACIÓN DE ESTANDARIZACIÓN MANUAL (Fase 2) ===")
print(f"Media de Z (esperada ≈ 0.0): {z_mean:.6f}")
print(f"Desviación estándar de Z (esperada ≈ 1.0): {z_std:.6f}")
print("\nPrimeros 5 registros con ingresos y su versión estandarizada:")
print(df_manual[['ingresos', 'ingresos_estandarizados']].head())

=== VERIFICACIÓN DE ESTANDARIZACIÓN MANUAL (Fase 2) ===
Media de Z (esperada ≈ 0.0): 0.000000
Desviación estándar de Z (esperada ≈ 1.0): 1.000000

Primeros 5 registros con ingresos y su versión estandarizada:
      ingresos  ingresos_estandarizados
0  80277.19986                 0.057712
1  57084.76993                -0.116194
2  55146.56536                -0.130727
3  60065.86646                -0.093840
4  52134.05204                -0.153316


---
# Fase 3: Enfoque 2 - Automatización con Pipelines (Scikit-Learn)

En esta sección se implementa un flujo automatizado y reproducible utilizando `scikit-learn`. Se vuelve a cargar el dataset original crudo para simular un escenario productivo donde nuevos lotes de datos deben transformarse de forma modular y sin fuga de datos (*data leakage*).

In [12]:
# Carga independiente del dataset crudo para la Fase 3
df_pipeline = pd.read_csv(dataset_path)

# Tratamiento inicial de anomalías lógicas de negocio
df_pipeline.loc[df_pipeline['edad'] < 0, 'edad'] = np.nan
df_pipeline.loc[df_pipeline['edad'] > 100, 'edad'] = np.nan
df_pipeline.loc[df_pipeline['ingresos'] < 0, 'ingresos'] = np.nan
df_pipeline.loc[df_pipeline['hijos'] < 0, 'hijos'] = np.nan

# Normalización básica de strings
for col in ['genero', 'ciudad', 'nivel_educacion']:
    df_pipeline[col] = df_pipeline[col].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan, '': np.nan})

# Definición de listas separadas para variables numéricas y categóricas
numeric_features = ['edad', 'ingresos', 'altura', 'hijos']
categorical_features = ['genero', 'ciudad', 'nivel_educacion']

print("Variables numéricas seleccionadas:", numeric_features)
print("Variables categóricas seleccionadas:", categorical_features)

Variables numéricas seleccionadas: ['edad', 'ingresos', 'altura', 'hijos']
Variables categóricas seleccionadas: ['genero', 'ciudad', 'nivel_educacion']


In [13]:
# Construcción de Pipelines y ColumnTransformer

# 1. Pipeline Numérico: SimpleImputer (mediana) + StandardScaler
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Pipeline Categórico: SimpleImputer (moda) + OneHotEncoder (dummies con puntos extra)
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. ColumnTransformer integrado
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ],
    remainder='drop'  # Ignora columnas como 'n'
)

print("Pipeline ColumnTransformer configurado exitosamente:")
print(preprocessor)

Pipeline ColumnTransformer configurado exitosamente:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['edad', 'ingresos', 'altura', 'hijos']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['genero', 'ciudad', 'nivel_educacion'])])


In [14]:
# Ejecución del pipeline con .fit_transform()
processed_data = preprocessor.fit_transform(df_pipeline)

# Extracción de los nombres de columnas resultantes
feature_names = preprocessor.get_feature_names_out()

# Creación del DataFrame procesado final
clean_column_names = [col.replace('num__', '').replace('cat__', '') for col in feature_names]
df_processed_final = pd.DataFrame(processed_data, columns=clean_column_names)

print(f"Pipeline ejecutado exitosamente.")
print(f"Dimensiones del dataset final: {df_processed_final.shape[0]} filas x {df_processed_final.shape[1]} columnas.")
print(f"Valores nulos residuales: {df_processed_final.isnull().sum().sum()}\n")
print("Primeros 5 registros del DataFrame procesado con Scikit-Learn:")
print(df_processed_final.head())

Pipeline ejecutado exitosamente.
Dimensiones del dataset final: 10000 filas x 26 columnas.
Valores nulos residuales: 0

Primeros 5 registros del DataFrame procesado con Scikit-Learn:
       edad  ingresos    altura     hijos  genero_F  genero_Femenino  genero_M  ...  nivel_educacion_Secundaria  nivel_educacion_bachiller  nivel_educacion_bachillers  nivel_educacion_master  nivel_educacion_mre  nivel_educacion_phd  nivel_educacion_sin  educacion
0  0.425764  0.057715  0.555699  0.257262       0.0              1.0       0.0  ...                         0.0                        0.0                         0.0                     0.0                  0.0                  1.0                             0.0
1  1.185201 -0.116199 -0.671249  0.595574       0.0              1.0       0.0  ...                         0.0                        0.0                         1.0                     0.0                  0.0                  0.0                             0.0
2 -0.158419 -0.130733 

---
# Pregunta de Reflexión Final y Justificación de Ingeniería

### ¿Cuál de los dos enfoques presentados considera más adecuado si el día de mañana la empresa recibe 50,000 registros nuevos cada hora para predecir en tiempo real? Justifique su respuesta.

**Respuesta: El Enfoque 2 (Automatización con Pipelines y ColumnTransformer de Scikit-Learn)** es el único viable y técnicamente adecuado para un entorno productivo de alta concurrencia e inferencia en tiempo real. 

Las razones arquitectónicas y metodológicas que fundamentan esta decisión son:

1. **Prevención Estricta de Fuga de Datos (*Data Leakage*):**
   - En el Enfoque 1 (Pandas manual), calcular `.median()` o `mean()` sobre el conjunto completo o recalcularlo en cada nuevo lote altera las referencias estadísticas con datos que el modelo no conocía durante el entrenamiento.
   - En el Enfoque 2, el objeto `Pipeline` aprende los parámetros estadísticos ($\mu$, $\sigma$, medianas y categorías del encoder) **únicamente en el conjunto de entrenamiento** (`.fit()`) y luego los aplica de forma congelada e inmutable (`.transform()`) sobre cada nuevo lote de 50,000 registros, garantizando validez estadística y rigor metodológico.

2. **Inferencia en Tiempo Real y Baja Latencia:**
   - La manipulación imperativa de Pandas implica operaciones lentas de interpretación en Python (`.apply()`, condicionales fila por fila, diccionarios manuales).
   - El `Pipeline` de Scikit-Learn está optimizado a nivel C/Cython y vectorizado con NumPy, procesando decenas de miles de filas en milisegundos. Además, puede serializarse en disco (`joblib.dump(preprocessor, 'pipeline.joblib')`) y cargarse instantáneamente en un microservicio de inferencia (por ejemplo, con FastAPI o Triton Inference Server).

3. **Consistencia Dimensional y Manejo de Categorías Nuevas:**
   - Si en un nuevo lote de 50,000 registros aparece una categoría imprevista (por ejemplo, una nueva ciudad como `"Cuenca"` o `"Loja"`), el `OneHotEncoder(handle_unknown='ignore')` la ignora asignando ceros sin detener la ejecución. En contraste, los mapeos manuales de Pandas generarían `NaN` accidentales o desalinearían las columnas necesarias por el algoritmo de Machine Learning (*shape mismatch error*).

4. **Mantenibilidad, MLOps y Reproducibilidad:**
   - Un `ColumnTransformer` encapsula todo el preprocesamiento en un solo objeto reutilizable, facilitando el versionado del modelo, la integración continua (CI/CD) y el monitoreo de deriva de datos (*Data Drift*).